# QRculus · 다변수 QR 판독 성공률 예측

Google Colab에서 **런타임 → 모두 실행**을 누르세요. GitHub의 원본 데이터를 다운로드하고 가림률·위치 탐지 패턴 가림률·가림 형태·오류 정정 등급으로 네 가지 분류 모델을 비교합니다.

**주의:** 위치 탐지 패턴의 정의가 3개 × 7×7 모듈(147칸)과 다를 경우 분모를 수정해야 합니다. 원본 데이터의 열 이름은 실행 시 확인합니다. 검증 지표는 Brier score, log loss 및 그룹별 성공률 MAE(%p)입니다.

In [ ]:
%pip -q install pandas numpy scikit-learn matplotlib requests

## 1. 데이터 다운로드 및 모델 학습

In [ ]:
"""QRculus multivariate model benchmark.
Run: pip install pandas numpy scikit-learn matplotlib requests
     python qrculus_multivariate.py
Reads the trial-level CSV, uses a payload-wise holdout, compares several models,
and saves predictions, metrics and a figure to qrculus_multivariate_results/.
"""
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, log_loss, brier_score_loss
import matplotlib.pyplot as plt

URL = 'https://raw.githubusercontent.com/hwanghj09/QRculus/main/results/pink_v2/raw/trials.csv'
OUT = Path('qrculus_multivariate_results'); OUT.mkdir(exist_ok=True)
file = Path('trials.csv')
if not file.exists():
    print('Downloading trial data from GitHub...')
    pd.read_csv(URL).to_csv(file, index=False)
df = pd.read_csv(file, low_memory=False)
print('Available columns:', ', '.join(df.columns))

def choose(*names):
    for n in names:
        if n in df.columns: return n
    return None

pct = choose('actual_pct','occlusion_pct','covered_pct','target_pct')
shape = choose('mask_type','shape')
level = choose('level','ecc_level','error_correction_level')
payload = choose('payload','payload_id','text','qr_text')
finder = choose('covered_finder','finder_covered','covered_finder_count','finder_coverage_pct','finder_coverage_ratio')
success = choose('success','is_success','decode_success','decoded_correctly','exact_match')
if not all([pct,shape,level,payload,finder,success]):
    raise ValueError('Could not identify required columns. Inspect printed headers and update choose() mappings.')

# Convert finder cell count to a coverage fraction. Standard QR Version 1 has
# three 7x7 finder patterns (147 core modules); adjust if dataset defines a
# different finder region or stores counts including separators.
X = df[[pct,finder,shape,level]].copy()
X.columns = ['occlusion_pct','finder_feature','mask_type','ecc_level']
if finder.endswith('_pct'): X['finder_feature'] /= 100
elif finder.endswith('_ratio'): pass
else: X['finder_feature'] /= 147
X['finder_feature'] = X['finder_feature'].clip(0,1)
y = df[success]
if y.dtype == object:
    y = y.astype(str).str.lower().map({'true':1,'false':0,'1':1,'0':0,'success':1,'failure':0})
y = pd.to_numeric(y, errors='coerce')
valid = X.notna().all(axis=1) & y.isin([0,1]) & df[payload].notna()
X,y,ids = X[valid],y[valid].astype(int),df.loc[valid,payload].astype(str)
# All repeats from one QR payload must stay on the same side of the split.
unique = np.array(sorted(ids.unique())); rng = np.random.default_rng(2026); rng.shuffle(unique)
holdout = set(unique[:max(1,round(len(unique)*.2))]); test = ids.isin(holdout).to_numpy()
num=['occlusion_pct','finder_feature']; cat=['mask_type','ecc_level']
pre=ColumnTransformer([('num',make_pipeline(SimpleImputer(strategy='median'),StandardScaler()),num),('cat',OneHotEncoder(handle_unknown='ignore',sparse_output=False),cat)])
models={
 'LogisticRegression':LogisticRegression(max_iter=2000,C=1),
 'RandomForest':RandomForestClassifier(n_estimators=250,min_samples_leaf=12,n_jobs=-1,random_state=2026),
 'ExtraTrees':ExtraTreesClassifier(n_estimators=250,min_samples_leaf=12,n_jobs=-1,random_state=2026),
 'HistGradientBoosting':HistGradientBoostingClassifier(max_iter=200,l2_regularization=3,random_state=2026),
}
results=[]; predictions=[]
for name,est in models.items():
    pipe=make_pipeline(pre,est);pipe.fit(X.loc[~test],y.loc[~test]);p=pipe.predict_proba(X.loc[test])[:,1]
    results.append({'model':name,'trial_brier':brier_score_loss(y.loc[test],p),'trial_log_loss':log_loss(y.loc[test],p),'trial_MAE_probability':mean_absolute_error(y.loc[test],p)})
    z=X.loc[test].copy();z['actual']=y.loc[test].to_numpy();z['predicted']=p;z['model']=name
    predictions.append(z)
metrics=pd.DataFrame(results).sort_values('trial_brier');metrics.to_csv(OUT/'metrics.csv',index=False)
pred=pd.concat(predictions);pred.to_csv(OUT/'predictions.csv',index=False)
print('\nTrial-level evaluation (lower is better):\n',metrics.to_string(index=False))
# Note: trial-level MAE of binary labels is not the same as MAE of grouped success rates.
# Produce group-level MAE as a more interpretable percentage-point error.
groupcols=['occlusion_pct','mask_type','ecc_level']
grp=pred.groupby(['model']+groupcols,as_index=False)[['actual','predicted']].mean()
grp['abs_error_pp']=100*(grp['actual']-grp['predicted']).abs()
score=grp.groupby('model',as_index=False)['abs_error_pp'].mean().sort_values('abs_error_pp')
score.to_csv(OUT/'group_mae.csv',index=False)
print('\nGrouped success-rate MAE (%p):\n',score.to_string(index=False))
fig,ax=plt.subplots(figsize=(8,4.5));ax.bar(score['model'],score['abs_error_pp']);ax.set_ylabel('Group MAE (percentage points)');ax.tick_params(axis='x',rotation=20);fig.tight_layout();fig.savefig(OUT/'model_comparison.png',dpi=180)
print('\nSaved results to:',OUT.resolve())


## 2. 결과 확인 및 다운로드
각 모델의 성능표와 비교 그래프를 확인하고 결과 ZIP 파일을 내려받습니다.

In [ ]:
from IPython.display import display, Image
from pathlib import Path
import shutil
display(metrics)
display(score)
display(Image(filename=str(OUT/'model_comparison.png')))
archive = shutil.make_archive('qrculus_multivariate_results', 'zip', root_dir='.', base_dir=str(OUT))
print('ZIP:', archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Colab 밖에서는 생성된 ZIP 파일을 직접 내려받으세요.')